In [ ]:
# imports
import pickle
import os
import re 
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score 
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import sklearn.metrics as metrics
import scipy as sp
import scipy.stats as stats
import warnings
from sklearn.pipeline import make_pipeline
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from toeplitzlda.classification import ToeplitzLDA
import logging
import math

# utils functions
from utils.preprocessing import all_have_same_condition, list_iterations, load_complete_session, inspect_session, get_n_epochs, get_iterations, get_n_iterations, load_session_chached
from utils.feature_extraction import get_jumping_means, epoch_vectorizer_channelprime
from utils.offline_evaluation import compare_auc_single_trial_interval, compute_auc_with_cv
from utils.online_simulation import online_simulation

# Turn off warnings (that most likely occur from ToeplitzLDA)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
mne.set_log_level('WARNING')



In [ ]:
X_train = [[3,20],
     [2,7],
     [1,10],
     [6,24],
     [1,7],
     [8,19],
     [1,8],
     [5,21]]
X_train = np.array(X_train) 
y_train = np.array([1, 2, 2, 1, 2, 1, 2, 1])

print(np.cov(X_train.T))

lda = LDA()
ldaclf1 = lda.fit(X_train, y_train)

print(ldaclf1.classes_)
print(ldaclf1.coef_)

S_w = ((np.cov(X_train[y_train==1].T)) + np.cov(X_train[y_train==2].T))/2
w = np.dot (np.linalg.pinv(S_w) , (np.mean(X_train[y_train==2],axis=0))-np.mean(X_train[y_train==1],axis=0))

ldaclf = make_pipeline(LDA(),)
ldaclf.fit(X_train,y_train)
print("-----LDA-----")
print((ldaclf.get_params()))
print(ldaclf.get_params().keys())
print((ldaclf.get_params().get("steps")))
print((ldaclf.get_params().get("verbose")))
print((ldaclf.get_params().get("lineardiscriminantanalysis")).coef_) # obtain w

slda = make_pipeline(LDA(solver='lsqr', shrinkage='auto'),)
slda.fit(X_train,y_train)

print("-----SLDA------")
print(slda.get_params()) 
print(slda.get_params().get("lineardiscriminantanalysis").coef_) # obtain w
print(slda.get_params().get("lineardiscriminantanalysis").__dict__)

nch = 2
btlda = make_pipeline(ToeplitzLDA(n_channels=nch),)
btlda.fit(X_train,y_train)

print("-----BTLDA-----")
print((btlda.get_params().get("toeplitzlda")).coef_) # cov matrix
print(btlda.get_params().get("toeplitzlda").intercept_) # cov matrix
print(btlda.get_params().get("toeplitzlda").__dict__) # cov matrix


